[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [APIs and JSON](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)

# Your First API Server


## What you will be able to do

Write an API with FastAPI: routes that answer `GET` requests, path and query parameters that arrive
as typed arguments, and a `404` for what does not exist. Run it in the notebook, send it requests,
and read the OpenAPI document and the documentation pages that FastAPI writes from your code.


## The idea

### The problem

Every request in this guide went to an API that someone else wrote. Behind each answer from the
practice API was code that read the request, found what it asked for, and chose a status code and a
body. To let other programs use your data, such as the readings a script collects or the results of
a model, you write that side.

Written with only the standard library, as the practice API is, a server reads every path and query
string by hand, turns text into numbers, checks each value, and writes every error itself. Its
documentation is written separately again, and nothing keeps the two together: a parameter added to
the code and forgotten in the document is invisible to anyone reading it.

### What an API server is

> An **API server** is a program that waits for HTTP requests, passes each one to the code written
> for its method and path, and sends that code's answer back as a response. **FastAPI** is a Python
> framework for writing one. A **route**, which FastAPI's documentation calls a path operation, is a
> function under a decorator such as `@app.get("/stations/{station_id}")`, which names the method and
> the path it answers. A name in braces in the path is a **path parameter**, and every other
> parameter of the function is a **query parameter**. FastAPI converts each to the type its hint
> names, answers `422` when a value will not convert, and sends what the function returns as JSON.
> From the same code it writes an **OpenAPI document** that describes every route, and serves
> documentation pages drawn from it. A separate server, **uvicorn**, runs the app and speaks HTTP.

### Why it works that way

- **A decorator puts a function on the app's list of routes.** `@app.get(...)` is a decorator, as
  the **Decorators** notebook in the **Object-Oriented Python** guide wrote them. It records the
  function with its method and path, and hands the function back unchanged, so a route can still be
  called like any other function.
- **Type hints do the parsing.** A request is text. `limit: int = 3` turns `?limit=5` into the
  number 5, gives 3 when the query leaves it out, and answers `422` for `?limit=five` before the
  function runs: the `422` the **Status Codes** notebook said FastAPI sends, seen from the server.
- **The app and the server are separate.** FastAPI decides what to answer, and uvicorn accepts the
  connections and speaks HTTP. Between them is ASGI, a standard interface between Python web servers
  and the applications they run, so the same app runs under any ASGI server.
- **An error is a response like any other.** Raising `HTTPException` stops a route and sends its
  status code with `{"detail": ...}`. A path that no route matches gets `404`, and a method that a
  path does not take gets `405` with an `Allow` header, without a line of code.
- **Documentation written from the code keeps up with it.** FastAPI builds its OpenAPI document from
  the routes and their type hints, so a parameter added to a function is in the document as soon as
  the app runs with it.

### Where you will meet this

FastAPI's home page quotes engineers at Microsoft, Uber, Netflix and Cisco on using it, among them
the team at Netflix that built Dispatch, its open-source crisis management orchestration framework,
with it. Flask and Django are two other Python web frameworks you will meet. This guide uses FastAPI
because its request and response models are the Pydantic models of the **Schemas and Validation**
notebook, and it publishes an OpenAPI document from them. Its page at `/docs` is Swagger UI, the
interactive documentation of Swagger's Petstore in the **Exploring an API** notebook. The
**Validating Requests** notebook gives an app a request body to check, the **A Complete API**
notebook rebuilds the practice API in FastAPI, and the **Hosting an API** notebook puts an app on a
public address.

### What this notebook covers

- An app, a route, and `practice_api.serve`, which runs an app in the background
- A path parameter, and a `404` raised with `HTTPException`
- Query parameters as typed arguments, with defaults, and the `422` for a value that does not convert
- What FastAPI answers with no route: `404`, and `405` with `Allow`
- The OpenAPI document at `/openapi.json`, and the pages at `/docs` and `/redoc`
- A title, a description, tags and docstrings, and where they appear
- A station API with three routes, checked against the practice API
- Five errors, from a route that answers another route's requests to a parameter named differently
  from its path

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import requests
from fastapi import FastAPI

import practice_api

app = FastAPI()


@app.get("/stations/{station_id}")
def station(station_id: str):
    return {"id": station_id, "name": station_id.title()}


app_url = practice_api.serve(app)
print(requests.get(f"{app_url}/stations/tromso", timeout=10).json())
```

```
{'id': 'tromso', 'name': 'Tromso'}
```

Four lines of FastAPI, and a server that answers `GET /stations/tromso` with JSON. The function never
saw the request: FastAPI took `tromso` from the path and passed it in.


## Setup

Eight imports, the last of them the practice API.

- `requests` sends requests to the apps in this notebook, as it sent them to the practice API
- `FastAPI` makes an app
- `HTTPException` is how a route answers with an error
- `urllib.request` fetches the practice API's code in Colab
- `Path` checks whether the practice API's code is already here
- `sys` tells this cell, and the cell that shows the documentation page, whether the notebook is
  running in Colab
- `importlib` reloads the practice API, so running this cell again uses its current code
- `practice_api` is the server this guide talks to, and its `serve` runs an app of your own

The apps in this notebook serve the practice API's own data, its stations in `practice_api.STATIONS`
and its readings in `practice_api.READINGS`.


In [1]:
import importlib
import sys
import urllib.request
from pathlib import Path

import requests
from fastapi import FastAPI, HTTPException

PRACTICE_API = "https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/notebooks/apis-and-json/practice_api.py"

if "google.colab" in sys.modules or not Path("practice_api.py").exists():
    urllib.request.urlretrieve(PRACTICE_API, "practice_api.py")    # in Colab, on every run

import practice_api
importlib.reload(practice_api)    # runs the file as it is now, not a copy imported earlier

BASE = practice_api.start()
print("The practice API is running at", BASE)


The practice API is running at http://127.0.0.1:8765


## Worked examples

### An app and a route

An app starts as `FastAPI()`. `@app.get(path)` above a function makes the function the route for
`GET` requests to that path, and `{station_id}` in the path is passed to the parameter with the same
name. What the function returns, a dictionary here, goes back as JSON. `practice_api.serve(app)` runs
the app in the background and returns its address:


In [2]:
app = FastAPI()


@app.get("/stations/{station_id}")
def station(station_id: str):
    return {"id": station_id, "name": station_id.title()}


app_url = practice_api.serve(app)
response = requests.get(f"{app_url}/stations/tromso", timeout=10)

print(app_url)
print(response.status_code, response.headers["Content-Type"], response.json())
print(station("oslo"))


http://127.0.0.1:8000
200 application/json {'id': 'tromso', 'name': 'Tromso'}
{'id': 'oslo', 'name': 'Oslo'}


The request went to port 8000, where uvicorn was listening, and FastAPI passed `tromso` to `station`,
which never saw the request itself. The decorator handed `station` back unchanged, so the last line
calls it directly, with no server involved.

A cell cannot run a server itself: a server waits for requests forever, and the cell would never
finish. `practice_api.serve` runs the app with uvicorn in a thread, as Setup's `start()` runs the
practice API, and serves one app at a time, so a cell that serves a new app replaces the one before,
on the same port. Outside a notebook, a terminal runs an app saved in `main.py` with
`uvicorn main:app`, where `main` is the file and `app` is the object in it, which the **Hosting an
API** notebook does.

### A path parameter, and a 404 of your own

A route decides its own errors. Raising `HTTPException` with a status code and a `detail` stops the
function there, and sends that status with the detail in the body. This route answers with a station
from `practice_api.STATIONS`, and with `404` for an id it does not have:


In [3]:
app = FastAPI()


@app.get("/stations/{station_id}")
def station(station_id: str):
    if station_id not in practice_api.STATIONS:
        raise HTTPException(status_code=404, detail=f"no station with id {station_id!r}")
    return practice_api.STATIONS[station_id]


app_url = practice_api.serve(app)
for station_id in ["tromso", "narvik"]:
    response = requests.get(f"{app_url}/stations/{station_id}", timeout=10)
    print(response.status_code, response.json())


200 {'id': 'tromso', 'name': 'Tromso', 'latitude': 69.65, 'longitude': 18.96}
404 {'detail': "no station with id 'narvik'"}


The body of the `404` is FastAPI's shape for an error, `{"detail": ...}`, where the practice API
sends `{"error": ...}`. Every API chooses its own, and a client written for one API, such as the
client in the **A Real Client** notebook, reads that API's shape. The **A Complete API** notebook
makes a FastAPI app answer in the practice API's shape. `HTTPException` is raised, not returned, and
Common errors shows what returning it does.

### Query parameters: typed arguments

Every parameter of a route that is not in its path is a query parameter. Its type hint says what to
convert the text to, and a default makes it optional. This route takes a required `station` and an
optional `limit`:


In [4]:
app = FastAPI()


@app.get("/readings")
def readings(station: str, limit: int = 3):
    """The latest readings from one station, newest first."""
    found = [reading for reading in practice_api.READINGS if reading["station"] == station]
    return found[::-1][:limit]


app_url = practice_api.serve(app)
for query in ["station=oslo", "station=oslo&limit=1", "station=svalbard", "limit=1", "station=oslo&limit=one"]:
    response = requests.get(f"{app_url}/readings?{query}", timeout=10)
    body = response.json()
    shown = body if response.status_code != 200 else [(reading["time"], reading["temperature_c"]) for reading in body]
    print(response.status_code, query, "|", shown)


200 station=oslo | [('2026-03-01T09:00Z', -4.2), ('2026-03-01T08:00Z', -4.9), ('2026-03-01T07:00Z', -5.5)]
200 station=oslo&limit=1 | [('2026-03-01T09:00Z', -4.2)]
200 station=svalbard | []
422 limit=1 | {'detail': [{'type': 'missing', 'loc': ['query', 'station'], 'msg': 'Field required', 'input': None}]}
422 station=oslo&limit=one | {'detail': [{'type': 'int_parsing', 'loc': ['query', 'limit'], 'msg': 'Input should be a valid integer, unable to parse string as an integer', 'input': 'one'}]}


`limit` arrived as the number 1, not the text `"1"`, and was 3 when the query left it out. Svalbard
has no readings, so the route sent an empty list: a query that matches nothing is not an error here.
The two `422`s came before the function ran, each with a `detail` list that says where the problem
is, as `["query", "station"]`, and what it is, in the form of Pydantic's errors in the **Schemas and
Validation** notebook.

### What FastAPI answers without a route

Some answers need no code. A path that no route matches, and a method that a path does not take, are
answered by FastAPI itself:


In [5]:
for method, path in [("GET", "/stations"), ("GET", "/readings/oslo"), ("POST", "/readings?station=oslo")]:
    response = requests.request(method, f"{app_url}{path}", timeout=10)
    print(method, path, "|", response.status_code, response.json(), "| Allow:", response.headers.get("Allow"))


GET /stations | 404 {'detail': 'Not Found'} | Allow: None
GET /readings/oslo | 404 {'detail': 'Not Found'} | Allow: None
POST /readings?station=oslo | 405 {'detail': 'Method Not Allowed'} | Allow: GET


This app has one route, `GET /readings`, so `/stations` and `/readings/oslo` got `404`, and a `POST`
to `/readings` got `405` with `Allow: GET`, the header the **What an API Is** notebook found on the
practice API. FastAPI's own `404` says `Not Found`, where the route's `404` named the station, which
is how a client tells a mistyped path from a missing station, as the **Status Codes** notebook did.

### The OpenAPI document FastAPI writes

FastAPI publishes a description of the app at `/openapi.json`, written from its routes the first time
it is asked for. It is an OpenAPI document, the kind the **Exploring an API** notebook read from the
practice API, whose document was written by hand:


In [6]:
document = requests.get(f"{app_url}/openapi.json", timeout=10).json()

print(document["openapi"], document["info"])
for path, operations in document["paths"].items():
    for method, operation in operations.items():
        print(method.upper(), path, "|", operation["summary"], "|", operation["description"])
        for parameter in operation["parameters"]:
            print("   ", parameter["name"], "in the", parameter["in"], "|", parameter["schema"],
                  "| required:", parameter["required"])
        print("    responses:", list(operation["responses"]))


3.1.0 {'title': 'FastAPI', 'version': '0.1.0'}
GET /readings | Readings | The latest readings from one station, newest first.
    station in the query | {'type': 'string', 'title': 'Station'} | required: True
    limit in the query | {'type': 'integer', 'default': 3, 'title': 'Limit'} | required: False
    responses: ['200', '422']


Everything in it came from the code: the path and the method from the decorator, `Readings` from the
function's name, the description from its docstring, each parameter's type from its hint, `limit`'s
default of 3, and a `422` response for the values FastAPI checks. The practice API's document sits
beside its code, and either one could change without the other. This one changes when the code does.

### The documentation pages

The same document draws two pages. `/docs` is Swagger UI, the interactive documentation of Swagger's
Petstore in the **Exploring an API** notebook, and `/redoc` is ReDoc, a second layout of the same
documentation. Both are HTML pages that load the document from `/openapi.json`:


In [7]:
for page in ["/docs", "/redoc"]:
    response = requests.get(f"{app_url}{page}", timeout=10)
    title = response.text.split("<title>")[1].split("</title>")[0]
    print(page, response.status_code, response.headers["Content-Type"], "|", title,
          "| loads /openapi.json:", "/openapi.json" in response.text)


/docs 200 text/html; charset=utf-8 | FastAPI - Swagger UI | loads /openapi.json: True
/redoc 200 text/html; charset=utf-8 | FastAPI - ReDoc | loads /openapi.json: True


On your own computer, open the page's address in a browser. In Colab the app runs on Colab's
machine, and `127.0.0.1` in your browser means your own computer, where nothing is listening.
`serve_kernel_port_as_iframe`, from `google.colab.output`, shows a page from a port on Colab's
machine inside the cell's output, so in Colab this cell shows the documentation there, and elsewhere
it says where to open it:


In [8]:
port = int(app_url.rsplit(":", 1)[1])

if "google.colab" in sys.modules:
    from google.colab import output
    output.serve_kernel_port_as_iframe(port, path="/docs", height="600")
else:
    print(f"Open {app_url}/docs in a browser on this computer.")


Open http://127.0.0.1:8000/docs in a browser on this computer.


In Swagger UI, open `GET /readings`, choose **Try it out**, fill in `station`, and choose
**Execute**. The page sends the request, and shows the response with the curl command that
sends it.

### A station API

The pieces of this notebook, in one app. It has a title, a description and a version for its
documentation, and tags that group its routes there. `GET /stations` lists the stations as the
practice API's `/stations` does, `GET /stations/{station_id}` sends one station or a `404`, and
`GET /stations/{station_id}/readings` sends a station's latest readings, with `limit` in the query,
or a `404` for a station that does not exist:


In [9]:
app = FastAPI(title="Weather stations", description="The practice API's stations, and their readings.", version="1.0.0")


@app.get("/stations", tags=["stations"])
def all_stations():
    """Every station, by id and name."""
    return [{"id": found["id"], "name": found["name"]} for found in practice_api.STATIONS.values()]


@app.get("/stations/{station_id}", tags=["stations"])
def one_station(station_id: str):
    """One station, with its coordinates."""
    if station_id not in practice_api.STATIONS:
        raise HTTPException(status_code=404, detail=f"no station with id {station_id!r}")
    return practice_api.STATIONS[station_id]


@app.get("/stations/{station_id}/readings", tags=["readings"])
def station_readings(station_id: str, limit: int = 3):
    """A station's latest readings, newest first."""
    one_station(station_id)                         # raises the 404 for a station that does not exist
    found = [reading for reading in practice_api.READINGS if reading["station"] == station_id]
    return found[::-1][:limit]


app_url = practice_api.serve(app)
for path in ["/stations/tromso/readings?limit=1", "/stations/svalbard/readings", "/stations/narvik/readings",
             "/stations/tromso/readings?limit=all"]:
    response = requests.get(f"{app_url}{path}", timeout=10)
    print(response.status_code, path, "|", response.json())

same = {path: requests.get(f"{app_url}{path}", timeout=10).json() == requests.get(f"{BASE}{path}", timeout=10).json()
        for path in ["/stations", "/stations/tromso"]}
print("the same answer as the practice API's:", same)

document = requests.get(f"{app_url}/openapi.json", timeout=10).json()
print(document["info"])
for path, operations in document["paths"].items():
    print("  GET", path, "| tags:", operations["get"]["tags"], "| responses:", list(operations["get"]["responses"]))


200 /stations/tromso/readings?limit=1 | [{'station': 'tromso', 'time': '2026-03-01T09:00Z', 'temperature_c': -6.3}]
200 /stations/svalbard/readings | []
404 /stations/narvik/readings | {'detail': "no station with id 'narvik'"}
422 /stations/tromso/readings?limit=all | {'detail': [{'type': 'int_parsing', 'loc': ['query', 'limit'], 'msg': 'Input should be a valid integer, unable to parse string as an integer', 'input': 'all'}]}
the same answer as the practice API's: {'/stations': True, '/stations/tromso': True}
{'title': 'Weather stations', 'description': "The practice API's stations, and their readings.", 'version': '1.0.0'}
  GET /stations | tags: ['stations'] | responses: ['200']
  GET /stations/{station_id} | tags: ['stations'] | responses: ['200', '422']
  GET /stations/{station_id}/readings | tags: ['readings'] | responses: ['200', '422']


### Where each part came from

| In the app | What it relies on | The section that showed it |
|---|---|---|
| `@app.get(...)` above each function | a decorator that makes a function a route | An app and a route |
| `raise HTTPException(status_code=404, ...)` | an error sent as a status code and a `detail` | A path parameter, and a 404 of your own |
| `one_station(station_id)` inside `station_readings` | a route that is still an ordinary function | An app and a route |
| `limit: int = 3` | a query parameter converted to its type, with a default | Query parameters: typed arguments |
| the `422` for `limit=all` | a value that does not convert, refused before the function runs | Query parameters: typed arguments |
| `title`, `description`, `version` and `tags` | documentation written from the app | The OpenAPI document FastAPI writes |
| `practice_api.serve(app)` | an app run in the background, one at a time | An app and a route |

Both answers compared with the practice API's matched it exactly, so for those two paths a client
could not tell the servers apart. Narvik's `404` was raised inside `one_station`, called from
`station_readings`: FastAPI's documentation notes that an `HTTPException` raised in any function
that a route calls ends the request there. Only `GET /stations`, which takes no parameters, has no
`422` among its responses.


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/15-your-first-api-server-solutions.ipynb).

**1.** Make an app with one route, `GET /ping`, that answers `{"status": "ok"}`. Serve it, and print
the status code and the body of a request to it.


In [10]:
# your code here


**2.** Make an app with a route, `GET /stations/{station_id}/coordinates`, that answers a station's
latitude and longitude as a list of two numbers, and a `404` with a detail for an id that is not a
station. Request it for `tromso` and for `bodo`, and print both answers.


In [11]:
# your code here


**3.** Make an app with a route, `GET /convert`, that takes a query parameter `celsius` as a `float`
and answers `{"celsius": ..., "fahrenheit": ...}`. Request `?celsius=-6.3` and `?celsius=cold`, and
print both answers.


In [12]:
# your code here


**4.** Give the route from task 3 a second query parameter, `digits`, an `int` with a default of 1,
and round the Fahrenheit value to that many digits. Request `?celsius=-6.3`, with and without
`digits=3`.


In [13]:
# your code here


**5.** Fetch the OpenAPI document of your app from task 4, and print every parameter of
`GET /convert`: its name, its type, and whether it is required.


In [14]:
# your code here


**6.** Make an app titled `Conversions`, with a route whose docstring says what it does. Print the
title and the route's description from the app's `/openapi.json`.


In [15]:
# your code here


## Common errors

### No error, and the wrong route: /stations/count answered by /stations/{station_id}


In [16]:
app = FastAPI()


@app.get("/stations/{station_id}")
def station(station_id: str):
    return {"id": station_id}


@app.get("/stations/count")
def station_count():
    return {"count": len(practice_api.STATIONS)}


app_url = practice_api.serve(app)
print(requests.get(f"{app_url}/stations/count", timeout=10).json())


{'id': 'count'}


FastAPI tries an app's routes in the order they were added, and `/stations/{station_id}` matches
`/stations/count`, with `count` as the station's id, so `station_count` never runs. FastAPI's
documentation says it plainly: order matters. Add the route with the fixed path first:


In [17]:
app = FastAPI()


@app.get("/stations/count")
def station_count():
    return {"count": len(practice_api.STATIONS)}


@app.get("/stations/{station_id}")
def station(station_id: str):
    return {"id": station_id}


app_url = practice_api.serve(app)
print(requests.get(f"{app_url}/stations/count", timeout=10).json())
print(requests.get(f"{app_url}/stations/oslo", timeout=10).json())


{'count': 4}
{'id': 'oslo'}


### No error, and 200 for a missing station: an HTTPException returned, not raised


In [18]:
app = FastAPI()


@app.get("/stations/{station_id}")
def station(station_id: str):
    if station_id not in practice_api.STATIONS:
        return HTTPException(status_code=404, detail=f"no station with id {station_id!r}")
    return practice_api.STATIONS[station_id]


app_url = practice_api.serve(app)
response = requests.get(f"{app_url}/stations/narvik", timeout=10)
print(response.status_code, response.json())


200 {'status_code': 404, 'detail': "no station with id 'narvik'", 'headers': None}


`return` sent the exception as a value, and FastAPI turned it into JSON like any other value, with a
`200`. A client that checks the status code takes Narvik for a station, and the 404 sits in the body,
where no client looks for a status. Only `raise` makes FastAPI send an `HTTPException`'s status:


In [19]:
app = FastAPI()


@app.get("/stations/{station_id}")
def station(station_id: str):
    if station_id not in practice_api.STATIONS:
        raise HTTPException(status_code=404, detail=f"no station with id {station_id!r}")
    return practice_api.STATIONS[station_id]


app_url = practice_api.serve(app)
response = requests.get(f"{app_url}/stations/narvik", timeout=10)
print(response.status_code, response.json())


404 {'detail': "no station with id 'narvik'"}


### No error, and the old answer: a route added again when its cell runs again


In [20]:
app = FastAPI()


@app.get("/greeting")
def greeting():
    return {"text": "Hello"}


# a later cell adds the same route again, with its answer edited
@app.get("/greeting")
def greeting():
    return {"text": "Hello from the stations"}


app_url = practice_api.serve(app)
print(requests.get(f"{app_url}/greeting", timeout=10).json())
print("routes for /greeting:", sum(route.path == "/greeting" for route in app.routes))


{'text': 'Hello'}
routes for /greeting: 2


A decorator adds a route every time it runs, and never replaces one. When a cell that adds a route to
an app made in an earlier cell runs again, edited, the app has two routes for the path, and the
first, with the old answer, is the one that matches. Make the app in the same cell as its routes, as
every cell in Worked examples does, so that running the cell again starts from an empty app:


In [21]:
app = FastAPI()


@app.get("/greeting")
def greeting():
    return {"text": "Hello from the stations"}


app_url = practice_api.serve(app)
print(requests.get(f"{app_url}/greeting", timeout=10).json())
print("routes for /greeting:", sum(route.path == "/greeting" for route in app.routes))


{'text': 'Hello from the stations'}
routes for /greeting: 1


### HTTPError: 500 Server Error: Internal Server Error for url: http://127.0.0.1:8000/stations/narvik


In [22]:
app = FastAPI()


@app.get("/stations/{station_id}")
def station(station_id: str):
    return practice_api.STATIONS[station_id]


app_url = practice_api.serve(app)
response = requests.get(f"{app_url}/stations/narvik", timeout=10)
response.raise_for_status()


HTTPError: 500 Server Error: Internal Server Error for url: http://127.0.0.1:8000/stations/narvik

`practice_api.STATIONS["narvik"]` raised `KeyError` inside the server. An exception in a route never
reaches the client: the app answered `500 Internal Server Error`, with only those words in the body,
and the traceback went to the server's log, which `practice_api.serve` keeps quiet in a notebook. A
missing station is not a failure of the server, so check for it and raise the `404` it is:


In [23]:
print(response.headers["Content-Type"], "|", response.text)

app = FastAPI()


@app.get("/stations/{station_id}")
def station(station_id: str):
    if station_id not in practice_api.STATIONS:
        raise HTTPException(status_code=404, detail=f"no station with id {station_id!r}")
    return practice_api.STATIONS[station_id]


app_url = practice_api.serve(app)
print(requests.get(f"{app_url}/stations/narvik", timeout=10).status_code)


text/plain; charset=utf-8 | Internal Server Error
404


### 422, Field required: a function parameter named differently from the path's


In [24]:
app = FastAPI()


@app.get("/stations/{station_id}")
def station(id: str):
    if id not in practice_api.STATIONS:
        raise HTTPException(status_code=404, detail=f"no station with id {id!r}")
    return practice_api.STATIONS[id]


app_url = practice_api.serve(app)
response = requests.get(f"{app_url}/stations/tromso", timeout=10)
print(response.status_code, response.json())


422 {'detail': [{'type': 'missing', 'loc': ['query', 'id'], 'msg': 'Field required', 'input': None}]}


FastAPI matches a path's names to the function's parameters by name. The path says `station_id` and
the function says `id`, so `id` is not in the path, and FastAPI took it for a query parameter, which
the request did not send: `loc` says `["query", "id"]`. Every station gets this `422`. Name the
parameter as the path does:


In [25]:
app = FastAPI()


@app.get("/stations/{station_id}")
def station(station_id: str):
    if station_id not in practice_api.STATIONS:
        raise HTTPException(status_code=404, detail=f"no station with id {station_id!r}")
    return practice_api.STATIONS[station_id]


app_url = practice_api.serve(app)
response = requests.get(f"{app_url}/stations/tromso", timeout=10)
print(response.status_code, response.json())


200 {'id': 'tromso', 'name': 'Tromso', 'latitude': 69.65, 'longitude': 18.96}


## Recap

- `FastAPI()` makes an app, and `@app.get(path)` makes a function the route for `GET` requests to
  that path. What the function returns is sent as JSON.
- A name in braces in the path is passed to the parameter with that name. Every other parameter is a
  query parameter, converted to the type its hint names, and optional when it has a default.
- A value that does not convert, or a required parameter left out, gets `422` with a `detail` list,
  before the route runs.
- `raise HTTPException(status_code=..., detail=...)` sends an error. FastAPI answers a path no route
  matches with `404`, and a method a path does not take with `405` and `Allow`.
- FastAPI writes an OpenAPI document at `/openapi.json` from the routes, their type hints and their
  docstrings, and draws `/docs` and `/redoc` from it.
- An app and the server that runs it are separate: `practice_api.serve` runs an app in a notebook,
  and `uvicorn main:app` runs one from a terminal.


## What is next

The **Validating Requests** notebook. Every value in this notebook arrived in a path or a query. That
notebook takes a JSON body, checks it against a Pydantic model with rules of its own, such as a
latitude between -90 and 90, and answers a body that breaks them with a `422` that says why.


---

&#8592; **Previous:** [A Real Client](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/14-a-real-client.ipynb)  &nbsp;·&nbsp;  [APIs and JSON Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)  &nbsp;·&nbsp;  **Next:** [Validating Requests](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/16-validating-requests.ipynb) &#8594;
